In [4]:
import os

os.environ["KAGGLE_API_TOKEN"] = open("../.env", encoding="utf-8").read().split("=", 1)[1].strip()

In [5]:
#import kagglehub

#path = kagglehub.competition_download(
#    "home-credit-default-risk",
#    output_dir="../data",
#)

#print("Path to competition files:", path)

In [6]:
import pandas as pd

app = pd.read_csv("../data/application_train.csv")

app.shape
app["TARGET"].value_counts(normalize=True)
app.isna().mean().sort_values(ascending=False).head(20)
app[["SK_ID_CURR", "TARGET", "AMT_INCOME_TOTAL", "AMT_CREDIT", "DAYS_BIRTH", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].head()

,SK_ID_CURR,TARGET,AMT_INCOME_TOTAL,AMT_CREDIT,DAYS_BIRTH,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3
0,100002,1,202500.0,406597.5,-9461,0.083037,0.262949,0.139376
1,100003,0,270000.0,1293502.5,-16765,0.311267,0.622246,NaN
2,100004,0,67500.0,135000.0,-19046,NaN,0.555912,0.729567
3,100006,0,135000.0,312682.5,-19005,NaN,0.650442,NaN
4,100007,0,121500.0,513000.0,-19932,NaN,0.322738,NaN


# ETL paso a paso: Home Credit Default Risk

Las tres primeras celdas descargan los datos y leen `application_train`. Desde aquí el notebook explica **qué hay en los datos, qué problemas tienen y qué decisión tomamos en cada caso**. Al final se ejecuta el ETL completo (`src/etl.py`) y se valida el resultado.

## 0. Contexto del problema (esto guía todas las decisiones)

- **Qué se predice:** `TARGET = 1` significa que el cliente tuvo dificultades de pago (atraso mayor a X días en alguna de las primeras cuotas). Es **clasificación binaria**.
- **Qué decisión apoya:** aprobar o rechazar un crédito. Un falso negativo (aprobar a quien no paga) cuesta mucho más que un falso positivo (rechazar a quien sí pagaba). Más adelante esto se traduce en elegir un **umbral de decisión** y no usar 0.5 por defecto.
- **Desbalance:** hay pocos positivos. Por eso `accuracy` no sirve y se usa AUC-ROC, PR-AUC y KS.
- **Una fila = una solicitud** (`SK_ID_CURR`). Las otras 6 tablas tienen **muchas filas por cliente**, así que hay que **agregarlas** antes de unir. Si se unen sin agregar, se duplican filas y se contamina el entrenamiento.
- **`application_test` no trae `TARGET`:** es para la competencia. Todas las transformaciones deben aplicarse igual a train y test.
- **Fuga de información (leakage):** todo lo que usemos debe existir **antes** de la decisión. Las tablas secundarias son historial previo a la solicitud, así que son válidas. Hay que tener cuidado con cualquier codificación que use `TARGET` (se haría dentro de la validación cruzada).
- **Variables sensibles:** `CODE_GENDER` y la edad (`DAYS_BIRTH`) pueden generar sesgo y problemas regulatorios en crédito real. Aquí las usamos porque es un ejercicio académico, pero es una decisión que hay que justificar.

## 1. Application: forma, tipos y balance de clases

In [7]:
print("Filas, columnas:", app.shape)
print("IDs duplicados:", app["SK_ID_CURR"].duplicated().sum())
print("\nTipos de columna:")
print(app.dtypes.astype(str).value_counts())
print("\nBalance de la variable objetivo:")
print(app["TARGET"].value_counts())
print(app["TARGET"].value_counts(normalize=True).round(4))

Filas, columnas: (307511, 122)
IDs duplicados: 0

Tipos de columna:
float64    65
int64      41
str        16
Name: count, dtype: int64

Balance de la variable objetivo:
TARGET
0    282686
1     24825
Name: count, dtype: int64
TARGET
0    0.9193
1    0.0807
Name: proportion, dtype: float64


**Qué buscar aquí:** que el ID sea único (lo es: no hay duplicados), cuántas columnas son numéricas y cuántas texto (las de texto necesitan codificación) y el desbalance. Hay unas 307 mil filas y 122 columnas, de las cuales unas 16 son texto. Solo cerca del **8%** son positivos: un modelo que prediga "todos pagan" tendría 92% de accuracy y sería inútil.

## 2. Valores nulos

In [8]:
nulos = app.isna().mean().sort_values(ascending=False)
print("Columnas con algún nulo:", (nulos > 0).sum(), "de", app.shape[1])
print("Columnas con más de 50% nulos:", (nulos > 0.5).sum())
nulos[nulos > 0].head(25).round(3)

Columnas con algún nulo: 67 de 122
Columnas con más de 50% nulos: 41


COMMONAREA_AVG              0.699
COMMONAREA_MODE             0.699
COMMONAREA_MEDI             0.699
NONLIVINGAPARTMENTS_MEDI    0.694
NONLIVINGAPARTMENTS_MODE    0.694
NONLIVINGAPARTMENTS_AVG     0.694
FONDKAPREMONT_MODE          0.684
LIVINGAPARTMENTS_AVG        0.684
LIVINGAPARTMENTS_MEDI       0.684
LIVINGAPARTMENTS_MODE       0.684
FLOORSMIN_MODE              0.678
FLOORSMIN_AVG               0.678
FLOORSMIN_MEDI              0.678
YEARS_BUILD_AVG             0.665
YEARS_BUILD_MODE            0.665
YEARS_BUILD_MEDI            0.665
OWN_CAR_AGE                 0.660
LANDAREA_MEDI               0.594
LANDAREA_AVG                0.594
LANDAREA_MODE               0.594
BASEMENTAREA_MODE           0.585
BASEMENTAREA_MEDI           0.585
BASEMENTAREA_AVG            0.585
EXT_SOURCE_1                0.564
NONLIVINGAREA_MODE          0.552
dtype: float64

In [9]:
# Los nulos se concentran en grupos de columnas. Veamos por sufijo/prefijo:
grupos = {
    "Edificio (_AVG/_MODE/_MEDI)": [c for c in app.columns if c.endswith(("_AVG", "_MODE", "_MEDI"))],
    "EXT_SOURCE": [c for c in app.columns if c.startswith("EXT_SOURCE")],
    "Buró (AMT_REQ_CREDIT_BUREAU_*)": [c for c in app.columns if c.startswith("AMT_REQ_CREDIT_BUREAU")],
}
for nombre, cols in grupos.items():
    print(f"{nombre}: {len(cols)} columnas, nulos promedio = {app[cols].isna().mean().mean():.1%}")

Edificio (_AVG/_MODE/_MEDI): 47 columnas, nulos promedio = 57.9%
EXT_SOURCE: 3 columnas, nulos promedio = 25.5%
Buró (AMT_REQ_CREDIT_BUREAU_*): 6 columnas, nulos promedio = 13.5%


**Interpretación:** un nulo no siempre es un dato perdido. Las ~47 columnas del **edificio** donde vive el cliente vienen casi todas nulas juntas (cuando no se tiene la información del inmueble). `EXT_SOURCE_1` tiene ~56% de nulos y `EXT_SOURCE_3` ~20%. Que falte el puntaje externo puede ser en sí una señal.

**Decisión:** **no imputamos** en el ETL. LightGBM/XGBoost aprovechan los nulos directamente. Si luego se prueba regresión logística, la imputación (mediana + indicador de nulo) va **dentro del pipeline del modelo**, ajustada solo con el fold de entrenamiento. Solo se eliminan columnas con más de 75% de nulos.

## 3. Problemas de calidad encontrados

### 3.1 `DAYS_EMPLOYED = 365243` (valor centinela)
`DAYS_EMPLOYED` cuenta días **hacia atrás** desde la solicitud, así que debería ser negativo. 365243 días son ~1000 años: es un código de "sin dato".

In [10]:
anom = app["DAYS_EMPLOYED"] == 365243
print("Filas con 365243:", anom.sum(), f"({anom.mean():.1%})")
print("Tasa de impago con 365243 :", app.loc[anom, "TARGET"].mean().round(4))
print("Tasa de impago sin 365243 :", app.loc[~anom, "TARGET"].mean().round(4))
print("\nTipo de ingreso en ese grupo:")
print(app.loc[anom, "NAME_INCOME_TYPE"].value_counts())
print("\nLas mismas filas tienen ORGANIZATION_TYPE == 'XNA'?",
      (anom == (app["ORGANIZATION_TYPE"] == "XNA")).all())

Filas con 365243: 55374 (18.0%)
Tasa de impago con 365243 : 0.054
Tasa de impago sin 365243 : 0.0866

Tipo de ingreso en ese grupo:
NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64

Las mismas filas tienen ORGANIZATION_TYPE == 'XNA'? True


**Hallazgo:** son ~55 mil filas (18%), casi todas **pensionados** (y algunos desempleados), y coinciden con `ORGANIZATION_TYPE = 'XNA'`. Su tasa de impago (~5.4%) es **menor** que la del resto (~8.7%).

**Decisión:** convertir 365243 a `NaN` (para no distorsionar promedios ni escalas) **y crear un indicador** `DAYS_EMPLOYED_ANOM`, porque el grupo se comporta distinto y esa información no se debe perder. Si solo borráramos el valor, perderíamos una señal real.

### 3.2 Categorías `XNA` / `Unknown`

In [11]:
print("CODE_GENDER:\n", app["CODE_GENDER"].value_counts(), "\n")
print("NAME_FAMILY_STATUS == 'Unknown':", (app["NAME_FAMILY_STATUS"] == "Unknown").sum())
print("ORGANIZATION_TYPE == 'XNA':", (app["ORGANIZATION_TYPE"] == "XNA").sum())

CODE_GENDER:
 CODE_GENDER
F      202448
M      105059
XNA         4
Name: count, dtype: int64 

NAME_FAMILY_STATUS == 'Unknown': 2
ORGANIZATION_TYPE == 'XNA': 55374


`XNA` significa "no aplica/no informado". Con 4 filas en `CODE_GENDER` no se puede aprender nada (esa tasa de impago de 0% sale de 4 filas, no es un patrón real). **Decisión:** pasarlas a `NaN` y que el one-hot cree una columna de "nulo".

### 3.3 Valores extremos

In [12]:
cols = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "CNT_CHILDREN", "OBS_30_CNT_SOCIAL_CIRCLE"]
app[cols].describe(percentiles=[.5, .99, .999]).round(1)

,AMT_INCOME_TOTAL,AMT_CREDIT,CNT_CHILDREN,OBS_30_CNT_SOCIAL_CIRCLE
count,307511.0,307511.0,307511.0,306490.0
mean,168797.9,599026.0,0.4,1.4
std,237123.1,402490.8,0.7,2.4
min,25650.0,45000.0,0.0,0.0
50%,147150.0,513531.0,0.0,0.0
99%,472500.0,1854000.0,3.0,10.0
99.9%,900000.0,2517300.0,4.0,17.0
max,117000000.0,4050000.0,19.0,348.0


El ingreso máximo es de **117 millones** frente a un percentil 99 de ~472 mil: un valor claramente atípico (probable error de captura o cliente excepcional). `CNT_CHILDREN` llega a 19 y `OBS_30_CNT_SOCIAL_CIRCLE` a 348.

**Decisión:** **no eliminamos outliers.** Los modelos de árboles son robustos a ellos, y borrar filas en datos de riesgo puede quitar justo los casos raros que importan. Si usamos un modelo lineal, se aplicará `log1p` o *winsorizing* dentro de su pipeline.

### 3.4 Columnas casi constantes

In [13]:
docs = [c for c in app.columns if c.startswith("FLAG_DOCUMENT_")]
app[docs].mean().sort_values().round(4).head(10)

FLAG_DOCUMENT_12    0.0000
FLAG_DOCUMENT_10    0.0000
FLAG_DOCUMENT_2     0.0000
FLAG_DOCUMENT_4     0.0001
FLAG_DOCUMENT_7     0.0002
FLAG_DOCUMENT_17    0.0003
FLAG_DOCUMENT_21    0.0003
FLAG_DOCUMENT_20    0.0005
FLAG_DOCUMENT_19    0.0006
FLAG_DOCUMENT_15    0.0012
dtype: float64

Varios `FLAG_DOCUMENT_*` son 1 en menos del 0.1% de los clientes: casi no aportan. **Decisión:** no los borramos todavía, pero en lugar de usarlos uno por uno creamos `DOCS_PROVIDED` (cuántos documentos entregó). La selección fina de variables se hace después con la importancia del modelo.

### 3.5 Unidades de tiempo
Todas las `DAYS_*` son **negativas y relativas a la fecha de solicitud** (no son fechas absolutas). Para que se lean mejor creamos `AGE_YEARS = -DAYS_BIRTH / 365`.

## 4. ¿Qué variables hay? (por grupo)

In [14]:
def grupo(c):
    if c in ("SK_ID_CURR", "TARGET"): return "ID / objetivo"
    if c.startswith("EXT_SOURCE"): return "Puntajes externos"
    if c.startswith("DAYS_"): return "Tiempo (días)"
    if c.startswith("AMT_REQ"): return "Consultas al buró"
    if c.startswith("FLAG_DOCUMENT"): return "Documentos"
    if c.startswith("FLAG_"): return "Contacto / banderas"
    if c.endswith(("_AVG", "_MODE", "_MEDI")): return "Edificio donde vive"
    if c.startswith(("AMT_", "NAME_CONTRACT")): return "Crédito / ingresos"
    if "SOCIAL_CIRCLE" in c: return "Círculo social"
    if c.startswith(("REG_", "LIVE_", "REGION")): return "Región / direcciones"
    if c.startswith("NAME_") or c.startswith("CODE_") or c.startswith("CNT_") or c in ("OCCUPATION_TYPE", "ORGANIZATION_TYPE"):
        return "Perfil del cliente"
    return "Otras"

resumen = pd.Series({c: grupo(c) for c in app.columns}).value_counts()
resumen

Edificio donde vive     47
Documentos              20
Perfil del cliente      10
Región / direcciones     9
Contacto / banderas      8
Consultas al buró        6
Crédito / ingresos       5
Tiempo (días)            5
Círculo social           4
Otras                    3
Puntajes externos        3
ID / objetivo            2
Name: count, dtype: int64

In [15]:
# Las 5 que más correlacionan con TARGET (ojo: correlación lineal, solo orientativa)
num = app.select_dtypes("number").drop(columns=["SK_ID_CURR"])
corr = num.corr()["TARGET"].drop("TARGET")
corr.reindex(corr.abs().sort_values(ascending=False).index).head(10).round(3)

EXT_SOURCE_3                  -0.179
EXT_SOURCE_2                  -0.160
EXT_SOURCE_1                  -0.155
DAYS_BIRTH                     0.078
REGION_RATING_CLIENT_W_CITY    0.061
REGION_RATING_CLIENT           0.059
DAYS_LAST_PHONE_CHANGE         0.055
DAYS_ID_PUBLISH                0.051
REG_CITY_NOT_WORK_CITY         0.051
FLAG_EMP_PHONE                 0.046
Name: TARGET, dtype: float64

Los más fuertes son `EXT_SOURCE_1/2/3` (correlación negativa de -0.16 a -0.18: mayor puntaje externo, menor riesgo). `DAYS_BIRTH` tiene correlación positiva (+0.08) porque los días son negativos: más edad, menos riesgo. Todas son débiles (|r| < 0.2): no hay una variable mágica, y por eso conviene combinar muchas. La correlación solo mide relaciones lineales; un árbol puede encontrar patrones que esta cifra no muestra.

## 5. Relación con la variable objetivo (sanity check)

Antes de modelar, comprobamos que los patrones tengan sentido de negocio.

In [16]:
for col in ["NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_EDUCATION_TYPE", "NAME_INCOME_TYPE"]:
    t = app.groupby(col)["TARGET"].agg(["mean", "size"]).round(4).sort_values("mean")
    print(f"\n{col}\n{t}")


NAME_CONTRACT_TYPE
                      mean    size
NAME_CONTRACT_TYPE                
Revolving loans     0.0548   29279
Cash loans          0.0835  278232

CODE_GENDER
               mean    size
CODE_GENDER                
XNA          0.0000       4
F            0.0700  202448
M            0.1014  105059

NAME_EDUCATION_TYPE
                                 mean    size
NAME_EDUCATION_TYPE                          
Academic degree                0.0183     164
Higher education               0.0536   74863
Incomplete higher              0.0848   10277
Secondary / secondary special  0.0894  218391
Lower secondary                0.1093    3816

NAME_INCOME_TYPE
                        mean    size
NAME_INCOME_TYPE                    
Businessman           0.0000      10
Student               0.0000      18
Pensioner             0.0539   55362
State servant         0.0575   21703
Commercial associate  0.0748   71617
Working               0.0959  158774
Unemployed            0.3636  

Los patrones son coherentes: a mayor educación, menor impago (educación superior ~5% frente a secundaria baja ~11%); los préstamos revolventes (tarjeta) incumplen menos que los de efectivo; y los hombres tienen más impago que las mujeres en esta muestra (aquí vuelve el punto de sesgo de la sección 0). **Tomar en cuenta los tamaños de grupo (`size`)**: una tasa de 0% con 4 filas no significa nada.

## 6. Tablas secundarias: cobertura y por qué se agregan

In [17]:
import pandas as pd
ids_total = set(app["SK_ID_CURR"]) | set(pd.read_csv("../data/application_test.csv", usecols=["SK_ID_CURR"])["SK_ID_CURR"])

for nombre in ["bureau", "previous_application", "POS_CASH_balance", "credit_card_balance", "installments_payments"]:
    ids = pd.read_csv(f"../data/{nombre}.csv", usecols=["SK_ID_CURR"])["SK_ID_CURR"]
    print(f"{nombre:24s} filas={len(ids):>10,}  clientes={ids.nunique():>7,}  "
          f"cobertura={ids.nunique() / len(ids_total):.0%}  filas/cliente={len(ids) / ids.nunique():.1f}")

bureau                   filas= 1,716,428  clientes=305,811  cobertura=86%  filas/cliente=5.6
previous_application     filas= 1,670,214  clientes=338,857  cobertura=95%  filas/cliente=4.9
POS_CASH_balance         filas=10,001,358  clientes=337,252  cobertura=95%  filas/cliente=29.7
credit_card_balance      filas= 3,840,312  clientes=103,558  cobertura=29%  filas/cliente=37.1
installments_payments    filas=13,605,401  clientes=339,587  cobertura=95%  filas/cliente=40.1


**Qué nos dice:**
- Cada cliente tiene entre ~5 filas (`bureau`, `previous_application`) y ~40 (`installments`, `credit_card`). Si uniéramos directo, una solicitud se repetiría decenas de veces. Por eso hacemos `groupby(SK_ID_CURR)` y calculamos resumen (media, máximo, mínimo, suma, conteo).
- `bureau_balance` no tiene `SK_ID_CURR`, solo `SK_ID_BUREAU`. Hay que agregarla **primero** a nivel de crédito del buró y luego a nivel de cliente (dos pasos).
- **Cobertura desigual:** `credit_card_balance` solo cubre el **29%** de los clientes. Un cliente sin filas ahí normalmente significa que **no tiene tarjeta de Home Credit**, no que falte un dato. Por eso los nulos se dejan como están (el modelo los lee como "sin historial") y no se rellenan con la media.
- Los clientes sin historial previo (~5-15%) son informativos por sí mismos: son clientes nuevos.

## 7. Qué hace el ETL (resumen de decisiones)

| Paso | Qué se hace | Por qué |
|---|---|---|
| Unir train y test | Se concatenan con una marca `IS_TRAIN` | Las mismas transformaciones para ambos |
| `DAYS_EMPLOYED` | 365243 a `NaN` + `DAYS_EMPLOYED_ANOM` | Centinela, pero con señal propia |
| `XNA`/`Unknown` | A `NaN` | Evitar categorías sin significado |
| Binarias `Y/N` | A 0/1 | Más compacto |
| Ratios | `CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`, `CREDIT_TERM`, etc. | Capacidad de pago, que es la pregunta del negocio |
| `EXT_SOURCE` | Media, mínimo, producto y promedio ponderado | Son las variables más predictivas |
| Tablas secundarias | Agregación por cliente (min/max/media/suma/conteo + reciente) | Quitar la relación 1 a N |
| Categóricas | One-hot (con columna de nulo) | Sin usar `TARGET`, así no hay fuga |
| Infinitos | `inf` a `NaN` | Divisiones entre 0 |
| Nulos | Se dejan; se descartan columnas con más del 75% | Los árboles los manejan |
| Tipos | `float32`/`int32` | Las tablas grandes caben en memoria |
| Salida | `train.parquet`, `test.parquet`, `feature_names.json` | Formato rápido y reproducible |

## 8. Ejecutar el ETL completo

La lógica está en `src/etl.py` (una función por tabla). Tarda unos 4 minutos; si ya existe `data/processed/`, se lee de ahí.

In [18]:
import sys
import warnings
from pathlib import Path

sys.path.insert(0, "..")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
from src.etl import run_etl

OUT = Path("../data/processed")
if (OUT / "train.parquet").exists():
    train = pd.read_parquet(OUT / "train.parquet")
    test = pd.read_parquet(OUT / "test.parquet")
else:
    train, test = run_etl()
print("train:", train.shape, "| test:", test.shape)

train: (307511, 606) | test: (48744, 605)


## 9. Validación del resultado

Un ETL no está terminado hasta comprobar que la salida es coherente.

In [19]:
assert train.index.is_unique and test.index.is_unique, "IDs duplicados"
assert "TARGET" in train.columns and "TARGET" not in test.columns
assert set(train.columns) - {"TARGET"} == set(test.columns), "train y test tienen columnas distintas"
assert not set(train.index) & set(test.index), "hay clientes en ambos conjuntos"
print("Chequeos estructurales OK")
print("Positivos en train:", train["TARGET"].mean().round(4))
print("Infinitos:", int(train.select_dtypes("number").isin([float("inf"), float("-inf")]).sum().sum()))
print("Memoria train: %.0f MB" % (train.memory_usage(deep=True).sum() / 1e6))

Chequeos estructurales OK
Positivos en train: 0.0807
Infinitos: 0
Memoria train: 603 MB


In [20]:
# ¿Se conserva la señal? Comparamos nulos y relación con TARGET de algunas variables creadas
nuevas = ["AGE_YEARS", "CREDIT_INCOME_RATIO", "ANNUITY_INCOME_RATIO", "CREDIT_TERM",
          "EXT_MEAN", "EXT_PROD", "DAYS_EMPLOYED_ANOM", "BUREAU_COUNT", "INS_DPD_MEAN", "PREV_COUNT"]
nuevas = [c for c in nuevas if c in train.columns]
pd.DataFrame({
    "nulos": train[nuevas].isna().mean().round(3),
    "corr_TARGET": train[nuevas].corrwith(train["TARGET"]).round(3),
})

,nulos,corr_TARGET
AGE_YEARS,0.000,-0.078
CREDIT_INCOME_RATIO,0.000,-0.008
ANNUITY_INCOME_RATIO,0.000,0.014
CREDIT_TERM,0.000,-0.032
EXT_MEAN,0.001,-0.222
EXT_PROD,0.644,-0.189
DAYS_EMPLOYED_ANOM,0.000,-0.046
BUREAU_COUNT,0.143,0.004
INS_DPD_MEAN,0.052,0.009
PREV_COUNT,0.054,0.020


**Cómo leerlo:** `EXT_MEAN` y `EXT_PROD` deberían correlacionar más fuerte con `TARGET` que cada `EXT_SOURCE` por separado (combinar ayuda). `BUREAU_COUNT` y `PREV_COUNT` con nulos indican clientes sin historial. Si alguna variable nueva tuviera 100% de nulos o correlación 0, algo falló en el ETL.

## 10. Preparación para modelar

Separamos `X` (variables) e `y` (`TARGET`). LightGBM no acepta ciertos caracteres en los nombres de columna (espacios, comas, `/`), y el one-hot los generó a partir de las categorías, así que los limpiamos.

**Métricas** (por el desbalance de ~8%, no usamos accuracy):
- **AUC-ROC:** probabilidad de que un cliente que no paga reciba un puntaje de riesgo mayor que uno que sí paga. 0.5 es azar y 1.0 es perfecto. Es la métrica de la competencia.
- **PR-AUC (average precision):** más sensible que el AUC cuando la clase positiva es rara. El azar equivale a la tasa de positivos (~0.08).
- **KS:** la máxima separación entre las distribuciones de buenos y malos pagadores. Es la métrica estándar en banca.

**Validación:** `StratifiedKFold` de 5 folds (mantiene el 8% de positivos en cada fold). Las predicciones *out-of-fold* (OOF) son siempre sobre clientes que el modelo **no vio** al entrenar, así que son una estimación honesta del desempeño.

In [21]:
import re

import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold

X = train.drop(columns="TARGET").copy()
y = train["TARGET"].astype(int)
X_test = test[X.columns].copy()

# nombres válidos para LightGBM (sin duplicar)
limpios = [re.sub(r"[^0-9a-zA-Z_]", "_", c) for c in X.columns]
assert len(set(limpios)) == len(limpios), "la limpieza de nombres creó duplicados"
X.columns = X_test.columns = limpios

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(X, y))


def ks_score(y_true, p):
    fpr, tpr, _ = roc_curve(y_true, p)
    return float(np.max(tpr - fpr))


def metricas(y_true, p):
    return {"AUC": roc_auc_score(y_true, p), "PR-AUC": average_precision_score(y_true, p), "KS": ks_score(y_true, p)}


print("X:", X.shape, "| X_test:", X_test.shape)
print("Positivos por fold:", [round(y.iloc[v].mean(), 4) for _, v in folds])

X: (307511, 605) | X_test: (48744, 605)
Positivos por fold: [np.float64(0.0807), np.float64(0.0807), np.float64(0.0807), np.float64(0.0807), np.float64(0.0807)]


## 11. Baseline: regresión logística

Es el modelo de referencia: simple, rápido e interpretable. Si el modelo principal no lo supera con claridad, no vale la pena la complejidad.

Un modelo lineal **no acepta nulos y es sensible a la escala**, así que el pipeline (que se ajusta **solo con el fold de entrenamiento** para no filtrar información del fold de validación) hace:
1. `SimpleImputer(median)`: rellena nulos con la mediana. Aquí sí imputamos, porque el modelo lo exige.
2. `QuantileTransformer(output_distribution="normal")`: convierte cada variable a una distribución normal según sus percentiles. Es robusto a colas extremas (el ingreso de 117 millones, las sumas de montos de las tablas agregadas).
3. `LogisticRegression` con regularización L2 (`C=0.01`, fuerte, porque hay más de 600 variables correlacionadas) y `class_weight="balanced"` para compensar el desbalance.

> **Lección de esta prueba:** la primera versión usaba `RobustScaler` y dio un AUC de apenas **0.58**, con advertencias de no convergencia. No era un problema de los datos sino de la escala: algunas variables agregadas tienen colas enormes que dominan la optimización. Al cambiar a `QuantileTransformer` el AUC subió a ~0.77. Si un baseline sale tan pobre, hay que sospechar primero del preprocesamiento, no del modelo.

In [22]:
import time

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer

warnings.filterwarnings("ignore", category=UserWarning)  # aviso de convergencia de lbfgs (max_iter=200)
oof_lr = np.zeros(len(X))
t0 = time.time()
for k, (tr, va) in enumerate(folds, 1):
    modelo = make_pipeline(
        SimpleImputer(strategy="median"),
        QuantileTransformer(n_quantiles=200, output_distribution="normal", subsample=100_000, random_state=0),
        LogisticRegression(C=0.01, class_weight="balanced", max_iter=200),
    )
    modelo.fit(X.iloc[tr], y.iloc[tr])
    oof_lr[va] = modelo.predict_proba(X.iloc[va])[:, 1]
    print(f"fold {k}: AUC = {roc_auc_score(y.iloc[va], oof_lr[va]):.4f}  ({time.time() - t0:.0f}s)")

res_lr = metricas(y, oof_lr)
print({k: round(v, 4) for k, v in res_lr.items()})

fold 1: AUC = 0.7710  (105s)
fold 2: AUC = 0.7818  (199s)
fold 3: AUC = 0.7736  (291s)
fold 4: AUC = 0.7791  (386s)
fold 5: AUC = 0.7702  (480s)
{'AUC': 0.7751, 'PR-AUC': 0.2587, 'KS': 0.413}


**Cómo leerlo:** el AUC por fold debería ser parecido entre folds (si varía mucho, la estimación es inestable). Este número es el piso: el modelo principal tiene que mejorarlo.

## 12. Modelo principal: LightGBM

**Por qué LightGBM:** es *gradient boosting* de árboles. Maneja nulos directamente (aprende hacia qué lado enviarlos, por eso no imputamos en el ETL), no necesita escalar, captura interacciones y no linealidades, y es rápido con 300 mil filas × 600 variables.

**Parámetros elegidos (todavía sin optimizar):**
- `learning_rate=0.05` con *early stopping*: se agregan árboles hasta que la validación deja de mejorar durante 100 rondas. Así no hay que fijar el número de árboles a mano.
- `colsample_bytree=0.5`, `subsample=0.8`: cada árbol ve un subconjunto de variables y filas, lo que reduce el sobreajuste.
- `num_leaves=31`, `min_child_samples=100`, regularización L1/L2: árboles pequeños y robustos.
- **Sin `scale_pos_weight`:** AUC, PR-AUC y KS miden el *ranking*, y reponderar las clases suele empeorar la calibración de las probabilidades. El costo asimétrico de los errores se maneja después, al elegir el **umbral de decisión**.

> **Nota:** el early stopping usa el fold de validación para decidir cuándo parar, así que el OOF es muy ligeramente optimista. Para la evaluación final se debería reservar un conjunto aparte.

In [23]:
import lightgbm as lgb

params = dict(
    objective="binary",
    learning_rate=0.05,
    n_estimators=5000,
    num_leaves=31,
    min_child_samples=100,
    colsample_bytree=0.5,
    subsample=0.8,
    subsample_freq=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

oof_lgb = np.zeros(len(X))
pred_test = np.zeros(len(X_test))
importancia = pd.DataFrame(index=X.columns)
mejores_iter = []
t0 = time.time()
for k, (tr, va) in enumerate(folds, 1):
    m = lgb.LGBMClassifier(**params)
    m.fit(
        X.iloc[tr], y.iloc[tr],
        eval_set=[(X.iloc[va], y.iloc[va])],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    oof_lgb[va] = m.predict_proba(X.iloc[va])[:, 1]
    pred_test += m.predict_proba(X_test)[:, 1] / len(folds)
    importancia[f"fold{k}"] = m.booster_.feature_importance("gain")
    mejores_iter.append(m.best_iteration_)
    print(f"fold {k}: AUC = {roc_auc_score(y.iloc[va], oof_lgb[va]):.4f}  "
          f"árboles = {m.best_iteration_}  ({time.time() - t0:.0f}s)")

res_lgb = metricas(y, oof_lgb)
print({k: round(v, 4) for k, v in res_lgb.items()})

c:\Users\dfgar\Documents\Unal\2026-2\optimizacion e inteligencia artificial\proyecto\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 1: AUC = 0.7885  árboles = 412  (54s)


c:\Users\dfgar\Documents\Unal\2026-2\optimizacion e inteligencia artificial\proyecto\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 2: AUC = 0.7992  árboles = 492  (99s)


c:\Users\dfgar\Documents\Unal\2026-2\optimizacion e inteligencia artificial\proyecto\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 3: AUC = 0.7899  árboles = 599  (153s)


c:\Users\dfgar\Documents\Unal\2026-2\optimizacion e inteligencia artificial\proyecto\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 4: AUC = 0.7934  árboles = 391  (198s)


c:\Users\dfgar\Documents\Unal\2026-2\optimizacion e inteligencia artificial\proyecto\venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


fold 5: AUC = 0.7904  árboles = 677  (255s)
{'AUC': 0.7922, 'PR-AUC': 0.2873, 'KS': 0.441}


## 13. Comparación de modelos

In [24]:
comparacion = pd.DataFrame({"Regresión logística": res_lr, "LightGBM": res_lgb}).T.round(4)
comparacion["mejora AUC vs baseline"] = (comparacion["AUC"] - comparacion.loc["Regresión logística", "AUC"]).round(4)
comparacion

,AUC,PR-AUC,KS,mejora AUC vs baseline
Regresión logística,0.7751,0.2587,0.413,0.0000
LightGBM,0.7922,0.2873,0.441,0.0171


## 14. ¿Qué variables usa el modelo?

Importancia por *gain*: cuánto reduce el error cada variable sumando todos los árboles. Se promedia entre folds. Sirve para **verificar que el ETL aportó** y para detectar posibles fugas de información (si una variable domina de forma sospechosa).

In [25]:
imp = (importancia.mean(axis=1) / importancia.mean(axis=1).sum()).sort_values(ascending=False)
top = pd.DataFrame({"importancia_%": (imp * 100).round(2)}).head(20)

def origen(c):
    for pref, nombre in [("BUREAU_", "bureau"), ("PREV_", "previous_application"), ("POS_", "POS_CASH"),
                         ("CC_", "credit_card"), ("INS_", "installments")]:
        if c.startswith(pref):
            return nombre
    return "application"

top["tabla"] = [origen(c) for c in top.index]
print(top)
por_tabla = imp.groupby([origen(c) for c in imp.index]).sum().mul(100).round(1).sort_values(ascending=False)
print("\nImportancia acumulada (%) por tabla de origen:\n", por_tabla)
print("\nVariables con importancia 0:", int((imp == 0).sum()), "de", len(imp))

                                importancia_%                 tabla
EXT_MEAN                                14.93           application
EXT_MIN                                  6.44           application
EXT_SOURCE_3                             2.87           application
BUREAU_CREDIT_DEBT_RATIO_MAX             2.28                bureau
CREDIT_TERM                              2.27           application
EXT_SOURCE_2                             1.82           application
GOODS_CREDIT_RATIO                       1.76           application
DAYS_EMPLOYED                            1.51           application
POS_CNT_INSTALMENT_FUTURE_MEAN           1.32              POS_CASH
EXT_PROD                                 1.17           application
BUREAU_CREDIT_DEBT_RATIO_MEAN            1.16                bureau
AMT_ANNUITY                              1.12           application
EXT_SOURCE_1                             1.05           application
INS_L12M_DPD_MEAN                        0.99   

**Cómo leerlo:** si las variables de `application` (en particular `EXT_*`) lideran y las tablas secundarias aportan un porcentaje relevante, el ETL cumplió su objetivo. Las variables con importancia 0 son candidatas a eliminar en una selección posterior.

## 15. Guardar resultados

In [26]:
resultados = Path("../data/processed")
pd.DataFrame({"TARGET": y, "oof_logreg": oof_lr, "oof_lgbm": oof_lgb}, index=X.index).to_parquet(resultados / "oof_predictions.parquet")
pd.DataFrame({"TARGET": pred_test}, index=X_test.index).to_csv(resultados / "test_pred_lgbm.csv")
print("Guardado OOF y predicciones de test en", resultados.resolve())

Guardado OOF y predicciones de test en C:\Users\dfgar\Documents\Unal\2026-2\optimizacion e inteligencia artificial\proyecto\data\processed


## 16. Hasta aquí y próximos pasos

Se llegó hasta el **baseline y el modelo principal**. Quedan para después, si se desea continuar:
1. **Optimización** de hiperparámetros (Optuna) y elección del **umbral de decisión** que minimice el costo esperado (un falso negativo cuesta mucho más que un falso positivo).
2. **Interpretabilidad** con SHAP y revisión de sesgo por género y edad.
3. Una evaluación final con un conjunto de prueba reservado.